# **N-BEATS**

Нейросеть для прогнозирования временных рядов на блоках “остаток + прогноз”.

Суть: складывает прогноз из нескольких блоков, каждый объясняет часть сигнала.

ŷ = ∑ᵦ gᵦ(x)


gᵦ — выход блока β; ŷ — итоговый прогноз.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
class NBeatsBlock(nn.Module):
    def init(self, in_len, n_features, hidden=256):
        super().init()
        self.fc = nn.Sequential(
            nn.Linear(in_len*n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.backcast = nn.Linear(hidden, in_len*n_features)

    def forward(self, x):
        b, l, f = x.shape
        h = self.fc(x.reshape(b, l*f))
        back = self.backcast(h).reshape(b, l, f)
        return x - back, h

class NBeatsCls(nn.Module):
    def init(self, seq_len, n_features, n_blocks=3, hidden=256, n_classes=3):
        super().init()
        self.blocks = nn.ModuleList([NBeatsBlock(seq_len, n_features, hidden) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        h_last = None
        for blk in self.blocks:
            x, h = blk(x)
            h_last = h
        return self.head(h_last)

def train_nbeats_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False)

        model = NBeatsCls(seq_len=seq_len, n_features=X_train.shape[1], n_blocks=3, hidden=256, n_classes=3)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=20, lr=1e-3)
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"N-BEATS",
            "model_family":"deep",
            "model_params":{"n_blocks":3,"hidden":256,"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })